# FMoE N3 Router/Wrapper Sanity Notebook

이 노트북은 primitive(a~e)와 wrapper(w1~w6)의 shape/확률/top-k 동작을 빠르게 점검하기 위한 sanity 체크입니다.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

ROOT = Path('/workspace/jy1559/FMoE/experiments')
sys.path.insert(0, str(ROOT))

from models.FeaturedMoE_N3.router_wrapper import (
    StageJointExpertRouter, StageGroupRouter, StageSharedIntraRouter,
    GroupConditionalIntraRouter, GroupScalarRouter, PrimitiveRoutingSpec,
    build_wrapper_module, required_primitives_for_wrapper,
)

torch.manual_seed(42)
G, C = 4, 3
E = G * C
B, S, D = 4, 6, 24


In [ ]:
stage_input = torch.randn(B, S, D)
group_input = torch.randn(B, S, G, D)
spec_dense = PrimitiveRoutingSpec(source='both', temperature=1.0, top_k=0)

a = StageJointExpertRouter(d_in=D, n_experts=E, d_hidden=16, dropout=0.0)(stage_input, spec_dense)
b = StageGroupRouter(d_in=D, n_groups=G, d_hidden=16, dropout=0.0)(stage_input, spec_dense)
c = StageSharedIntraRouter(d_in=D, n_experts_per_group=C, d_hidden=16, dropout=0.0)(stage_input, spec_dense)
d = GroupConditionalIntraRouter(d_in=D, n_groups=G, n_experts_per_group=C, d_hidden=16, dropout=0.0)(group_input, spec_dense)
e = GroupScalarRouter(d_in=D, n_groups=G)(group_input, spec_dense)

print('A', a['probs'].shape, a['probs'].sum(dim=-1).mean().item())
print('B', b['probs'].shape, b['probs'].sum(dim=-1).mean().item())
print('C', c['probs'].shape, c['probs'].sum(dim=-1).mean().item())
print('D', d['probs'].shape, d['probs'].sum(dim=-1).mean().item())
print('E', e['probs'].shape, e['probs'].sum(dim=-1).mean().item())


In [ ]:
wrappers = ['w1_flat', 'w2_a_plus_d', 'w3_bxc', 'w4_bxd', 'w5_exd', 'w6_bxd_plus_a']
primitives = {
    'a_joint': a,
    'b_group': b,
    'c_shared': c,
    'd_cond': d,
    'e_scalar': e,
}

for name in wrappers:
    wrapper = build_wrapper_module(name)
    out = wrapper(
        primitives={k: v for k, v in primitives.items() if k in required_primitives_for_wrapper(name)},
        stage_temperature=1.0,
        n_groups=G,
        n_experts_per_group=C,
        params={'alpha_d': 1.0, 'alpha_struct': 1.0, 'alpha_a': 1.0},
    )
    probs = torch.softmax(out['scaled_logits'], dim=-1)
    active = (probs > 0).sum(dim=-1).float().mean().item()
    print(name, 'required=', required_primitives_for_wrapper(name), 'mean_active=', round(active, 3))


In [ ]:
# top-k 효과 시각화: d_cond top_k=1이면 그룹별 1개 expert만 활성
spec_sparse = PrimitiveRoutingSpec(source='both', temperature=1.0, top_k=1)
d_sparse = GroupConditionalIntraRouter(d_in=D, n_groups=G, n_experts_per_group=C, d_hidden=16, dropout=0.0)(group_input, spec_sparse)
active = (d_sparse['probs'] > 0).sum(dim=-1)
print('d_cond top_k=1 active count (min,max)=', int(active.min()), int(active.max()))

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.imshow(d_sparse['probs'][0].reshape(G, C).detach().cpu(), cmap='viridis', aspect='auto')
ax.set_title('Sample0 D_COND probs (G x C)')
ax.set_xlabel('expert-in-group')
ax.set_ylabel('group')
plt.tight_layout()
plt.show()
